# Custom Dataset Evaluation Tutorial

This notebook demonstrates how to evaluate your model against **custom query-answer datasets**.

**What you'll learn:**
- Loading evaluation datasets from CSV files
- Defining async model callables for evaluation
- Running evaluations with an LLM judge
- Analyzing results by category and difficulty

## Setup

First, let's install dependencies and set up API keys.

In [19]:
%pip install -q tavily-agent-toolkit tavily-python langchain langchain-openai langchain-community langgraph pandas


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY:\n")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY:\n")

In [21]:
# Import the evaluation framework
from tavily_agent_toolkit import ModelConfig, ModelObject
from tavily_agent_toolkit.evals import (
    DatasetEvaluator,
    CSVDataset,
    InMemoryDataset,
    DatasetItem,
    evaluate_dataset,
)

# Configure the judge model (used to grade correctness)
judge_config = ModelConfig(
    model=ModelObject(
        model="gpt-5-mini",
        api_key=os.environ["OPENAI_API_KEY"],
    ),
)

## Creating a Dataset

You can create datasets in two ways:
1. Load from a CSV file
2. Create programmatically in memory

### CSV Format

The CSV format requires:
- `query`: The question to ask your model
- `expected_answer`: The correct answer for evaluation

Optional columns:
- `category`: Group questions by type (e.g., "factual", "reasoning")
- `difficulty`: Difficulty level (e.g., "easy", "medium", "hard")
- `metadata`: JSON string with additional data

In [22]:
# Create a sample dataset programmatically
# These are all current events that require web search to answer correctly
sample_items = [
    # Technology - Programming Languages & Frameworks
    DatasetItem(
        query="What is the latest stable version of Python?",
        expected_answer="3.14.3",
    ),
    DatasetItem(
        query="What is the current LTS version of Node.js?",
        expected_answer="24",
    ),
    DatasetItem(
        query="What is the codename for Node.js 24?",
        expected_answer="Krypton",
    ),
    # AI News
    DatasetItem(
        query="Which AI model was used to plan the first AI-assisted drive on Mars?",
        expected_answer="Claude",
    ),
    DatasetItem(
        query="Which Chinese AI company released the R1 model in January 2025?",
        expected_answer="DeepSeek",
    ),
    DatasetItem(
        query="What was Anthropic's valuation after its $13 billion Series F round in 2025?",
        expected_answer="$183 billion",
    ),
    # Current Events
    DatasetItem(
        query="Who is the current Prime Minister of Canada?",
        expected_answer="Mark Carney",
    ),
    DatasetItem(
        query="What film won Best Picture at the 2025 Academy Awards?",
        expected_answer="Anora",
    ),
    DatasetItem(
        query="Who won Album of the Year at the 2025 Grammy Awards?",
        expected_answer="Beyoncé",
    ),
    DatasetItem(
        query="Who won Super Bowl LIX in 2025?",
        expected_answer="Philadelphia Eagles",
    ),
]

dataset = InMemoryDataset(sample_items)
print(f"Dataset loaded with {len(dataset)} items")

Dataset loaded with 10 items


In [23]:
# Save the dataset to CSV so we can demonstrate loading from file
import pandas as pd

rows = [{"query": item.query, "expected_answer": item.expected_answer, "category": item.category or "", "difficulty": item.difficulty or ""} for item in dataset]
pd.DataFrame(rows).to_csv("sample_dataset.csv", index=False)
print("Saved sample_dataset.csv")

Saved sample_dataset.csv


## Defining Your Model

Your model must be an async callable with this signature:

```python
async def model(query: str) -> str:
    # Return the answer as a string
    ...
```

Let's create a few example models to compare — including a **tool-calling agent** that can decide when to search the web.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent

# Initialize LangChain components
llm = ChatOpenAI(model="gpt-5-mini")
tavily_tool = TavilySearchResults(max_results=5, search_depth="advanced")

# Model 1: Simple LLM (no search)
async def llm_only_model(query: str) -> str:
    """Answer using only the LLM's knowledge."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer questions concisely and accurately."),
        ("user", "{query}")
    ])
    chain = prompt | llm
    response = await chain.ainvoke({"query": query})
    return response.content


# Model 2: Tool-calling Agent with Tavily Search
agent = create_agent(
    llm,
    tools=[tavily_tool],
    system_prompt="Answer questions concisely and accurately. Use the search tool to find up-to-date information when needed.",
)

async def agent_model(query: str) -> str:
    """Answer using an agent that can search the web with Tavily."""
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": query}]}
    )
    return result["messages"][-1].content


print("Models defined!")

Models defined!


## Running Evaluation

Now let's evaluate our models against the dataset.

In [25]:
# Create the evaluator
evaluator = DatasetEvaluator(
    judge_model_config=judge_config,
    parallel=True,  # Run evaluations in parallel
    max_concurrency=5,  # Limit concurrent API calls
)

print("Evaluator ready!")

Evaluator ready!


In [26]:
# Evaluate the LLM-only model
print("Evaluating LLM-only model...")
llm_result = await evaluator.evaluate(dataset, llm_only_model)

print(f"\nLLM-Only Model Results:")
print(f"  Accuracy: {llm_result.accuracy:.1%}")
print(f"  Correct: {llm_result.correct_count}/{llm_result.total_count}")

Evaluating LLM-only model...

LLM-Only Model Results:
  Accuracy: 10.0%
  Correct: 1/10


In [27]:
# Evaluate the Agent model with Tavily web search
print("Evaluating Agent model...")
agent_result = await evaluator.evaluate(dataset, agent_model)

print(f"\nAgent Model Results:")
print(f"  Accuracy: {agent_result.accuracy:.1%}")
print(f"  Correct: {agent_result.correct_count}/{agent_result.total_count}")

Evaluating Agent model...

Agent Model Results:
  Accuracy: 100.0%
  Correct: 10/10


In [28]:
# Compare the models
print("\n" + "="*50)
print("Model Comparison")
print("="*50)
print(f"LLM-Only: {llm_result.accuracy:.1%} accuracy")
print(f"Agent:    {agent_result.accuracy:.1%} accuracy")
print("="*50)


Model Comparison
LLM-Only: 10.0% accuracy
Agent:    100.0% accuracy


## Using the Convenience Function

For quick evaluations, use the `evaluate_dataset` function directly.

In [32]:
# Quick evaluation with the convenience function
result = await evaluate_dataset(
    dataset=dataset,
    model=agent_model,
    judge_model_config=judge_config,
    parallel=True,
)

print(f"Quick eval accuracy: {result.accuracy:.1%}")

Quick eval accuracy: 100.0%


## Exporting Results

Results can be exported in various formats for reporting.

In [ ]:
import json

# Export as JSON
result_dict = agent_result.to_dict()
print("Result as JSON:")
print(json.dumps({
    "accuracy": result_dict["accuracy"],
    "correct_count": result_dict["correct_count"],
    "total_count": result_dict["total_count"],
    "by_category": result_dict.get("by_category", {}),
}, indent=2))

In [ ]:
# Export to CSV
df = agent_result.to_dataframe()
df.to_csv("evaluation_results.csv", index=False)
print("Saved evaluation_results.csv")
print(df.head())

## Understanding the Judge

The evaluation uses an LLM judge to determine if answers are correct.

### Grading Criteria

- **CORRECT**: The predicted answer conveys the same essential information as the expected answer. Minor differences in phrasing, formatting, or additional context are acceptable.

- **INCORRECT**: The predicted answer is factually wrong, contradicts the expected answer, misses key information, or fails to answer the question.

### Tips for Good Evaluation

1. **Write clear expected answers** - Be specific about what constitutes a correct answer
2. **Use consistent formatting** - If you expect "Paris", "paris" will still be correct (semantic matching)
3. **Include variations** - Multiple acceptable answers can be captured in the expected answer
4. **Test your dataset** - Run a few manual checks to ensure the judge grades reasonably

In [30]:
# View judge usage statistics
print("Judge Usage:")
print(f"  LLM calls: {agent_result.usage.judge_llm_calls}")
print(f"  Input tokens: {agent_result.usage.judge_input_tokens}")
print(f"  Output tokens: {agent_result.usage.judge_output_tokens}")
print(f"  Total tokens: {agent_result.usage.judge_total_tokens}")
print(f"  Response time: {agent_result.usage.eval_response_time:.2f}s")

Judge Usage:
  LLM calls: 10
  Input tokens: 4032
  Output tokens: 1302
  Total tokens: 5334
  Response time: 32.22s
